### If necessary, install Cursor Extensions

1. From the View menu, select Extensions
2. Search for Python
3. Click on "Python" made by "ms-python" and select Install if not already installed
4. Search for Jupyter
5. Click on "Jupyter" made by "ms-toolsai" and select Install if not already installed


### Next Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

Any problems with this? Head over to the troubleshooting.

### Note: you'll need to set the Kernel with every notebook..

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  


In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = None
base_url = None
run_with = 1
temperature = 0.5 # lower for more precise answers
max_tokens = 1000 # lower for faster responses
# Coding settings
#temperature = 0.2 # lower for more precise answers
#max_tokens = 2000 # lower for faster responses

if run_with == 1:
    api_key = os.getenv('LMSTUDIO_API_KEY')
    base_url = os.getenv('LMSTUDIO_BASE_URL')
    ai_model = "openai/gpt-oss-20b"
    #ai_model = "mistralai/devstral-small-2-2512"
elif run_with == 2:
    api_key = os.getenv('OPENAI_API_KEY')
    base_url = os.getenv('OPENAI_BASE_URL')
    ai_model = "gpt-5-nano"
elif run_with == 3:
    api_key = os.getenv('DEEPSEEK_API_KEY')
    base_url = os.getenv('DEEPSEEK_BASE_URL')
    ai_model = "deepseek-chat"
elif run_with == 4:
    base_url = os.getenv('MISTRAL_BASE_URL')
    api_key = os.getenv('MISTRAL_API_KEY')
    ai_model = "mistral-large-2512"

print(api_key)
print(base_url)
print(ai_model)
openai = OpenAI(base_url=base_url, api_key=api_key)

lmstudio
http://127.0.0.1:1234/v1
openai/gpt-oss-20b


# Let's make a quick call to a Frontier model to get started, as a preview!

In [60]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [ ]:

response = openai.chat.completions.create(model=ai_model, messages=messages, temperature=temperature, max_tokens=max_tokens)
response.choices[0].message.content

## OK onwards with our first project

In [61]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Conne

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [62]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [63]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model=ai_model, messages=messages, temperature=temperature, max_tokens=max_tokens)
response.choices[0].message.content

## And now let's build useful messages for the model, using a function

In [65]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [66]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n.

## Time to bring it together - the API for OpenAI is very simple!

In [67]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = ai_model,
        messages = messages_for(website),
        max_tokens = max_tokens,
        temperature = temperature
    )
    return response.choices[0].message.content

In [68]:
summarize("https://edwarddonner.com")

'**Edward Donner’s personal playground**\n\n- **Who is he?** A self‑proclaimed code‑wizard, DJ (in a *very* amateur capacity), and LLM tinkerer who still thinks “Hacker News” is the pinnacle of intellectual discourse.\n- **What’s he up to?** Co‑founder/CTO at Nebula.io, where they’re apparently using AI to help recruiters find “talent” while also claiming a patented matching model that *really* makes sense. Also bragging about an old startup (untapt) that got acquired in 2021—because who doesn’t love a good exit story?\n- **Fun side projects:** A game called *Connect Four* where LLMs battle for diplomatic supremacy, and a handful of blog posts titled “AI Live Event,” “Gen AI on AWS at scale,” and the like. Basically, he’s trying to convince you that AI is everywhere—just don’t ask him how it actually works.\n- **Contact info:** Email (ed@edwarddonner.com), LinkedIn, Twitter, Facebook… basically every social platform except *the* one where real developers hang out.\n\nBottom line: If yo

In [69]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [70]:
display_summary("https://edwarddonner.com")

# Edward Donner’s Personal Webpage

> **“I’m Ed, I code, DJ, and occasionally pretend to understand Hacker News.”**

- **About**: A self‑proclaimed tech wizard who also “loves” DJing (but is *badly* out of practice). He runs Nebula.io, an AI recruiting platform that’s supposedly “massively positive” – because nothing says life‑changing like matching people to jobs with patented LLMs.  
- **Career Highlights**: Former founder/CEO of a startup that got bought in 2021 (because why not brag about acquisition dates?).  
- **Current Projects**: A game called *Outsmart* where LLMs duel in “diplomacy and deviousness” – basically the AI version of a high‑stakes poker night.  
- **News & Posts**: A handful of dated blog posts about AI events, an AWS talk, a curriculum, and an executive briefing. All very generic, no real meat.  
- **Contact**: Email, LinkedIn, Twitter, Facebook – choose your poison.  

Bottom line: Ed is a tech‑savvy self‑promoter with a side hustle in AI recruiting and a hobby in DJing. If you’re looking for cutting‑edge AI talent solutions or an AI battle arena, he’s the guy to follow (or at least click through).

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [71]:
display_summary("https://cnn.com")

# CNN – The *“Breaking News”* that’s actually just a giant ad‑testing playground

- **Headline**: “Breaking News, Latest News and Videos | CNN” – because nothing says *news* like a page full of feedback forms.
- **Core content**: A never‑ending list of ad‑performance questions (“Did the video load? Was the audio too loud?”). If you’re into statistics about banner ads, this is your place.
- **Navigation**: Every category (US, World, Politics, etc.) is listed twice—once for “Watch” and once for “Listen.” Because apparently CNN thinks we need to hear the same thing in two different ways.
- **User account section**: Repeats “Sign in / My Account” multiple times. Maybe they’re trying to get you to sign in *again* after each ad fails.
- **No actual news**: The only stories here are about how annoying ads can be. If you were hoping for a headline about the Ukraine‑Russia war or the latest tech trend, you’ll have to look elsewhere.

Bottom line: CNN’s current homepage is less “breaking news” and more “break‑the‑ads.”

In [47]:
display_summary("https://anthropic.com")

**Anthropic: Where AI Meets Safety (and a Side of Existential Dread)**

Welcome to **Anthropic**, the AI research lab that’s *so* committed to safety, they’ll remind you *three times* to log in to Claude before you even get to the good stuff. Their mission? To build AI that won’t accidentally turn us all into paperclips—or at least, to *try* not to.

### **The Main Event: Claude Opus 4.5**
Anthropic just dropped **Claude Opus 4.5**, which they *very humbly* claim is the "best model in the world" for coding, agents, and enterprise workflows. (No pressure, other AI models.) They’re also hyping up "advanced tool use" on their developer platform—because nothing says "safety first" like giving AI more tools to play with.

### **The Vibe**
- **"We’re building AI for humanity’s long-term well-being!"** (Translation: *Please don’t let our AI become Skynet.*)
- **"Bold steps forward and intentional pauses!"** (Translation: *We’re moving fast but pretending to be responsible.*)
- **Three identical "Read announcement" buttons** (Translation: *We really, really want you to click this.*)

### **News & Announcements**
- **Claude Opus 4.5 is here**, and it’s *apparently* a big deal. (We’ll believe it when it stops hallucinating.)
- **Developer tools are getting an upgrade**, because why let humans have all the fun?

### **Final Verdict**
Anthropic: *Trying* to be the responsible older sibling of AI labs, while still releasing shiny new models that might or might not respect our autonomy. **10/10 for effort, 5/10 for subtlety.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise, you experienced calling the Cloud API of a Frontier Model (a leading model at the frontier of AI) for the first time. We will be using APIs like OpenAI at many stages in the course, in addition to building our own LLMs.

More specifically, we've applied this to Summarization - a classic Gen AI use case to make a summary. This can be applied to any business vertical - summarizing the news, summarizing financial performance, summarizing a resume in a cover letter - the applications are limitless. Consider how you could apply Summarization in your business, and try prototyping a solution.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you continue - now try yourself</h2>
            <span style="color:#900;">Use the cell below to make your own simple commercial example. Stick with the summarization use case for now. Here's an idea: write something that will take the contents of an email, and will suggest an appropriate short subject line for the email. That's the kind of feature that might be built into a commercial email tool.</span>
        </td>
    </tr>
</table>

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(

## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or similar, then feel free to improve the Website class to use them. In the community-contributions folder, you'll find an example Selenium solution from a student (thank you!)

# Sharing your code

I'd love it if you share your code afterwards so I can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and I'll merge your changes.

If you're not an expert with git (and I am not!) then GPT has given some nice instructions on how to submit a Pull Request. It's a bit of an involved process, but once you've done it once it's pretty clear. As a pro-tip: it's best if you clear the outputs of your Jupyter notebooks (Edit >> Clean outputs of all cells, and then Save) for clean notebooks.

Here are good instructions courtesy of an AI friend:  
https://chatgpt.com/share/677a9cb5-c64c-8012-99e0-e06e88afd293